<a href="https://colab.research.google.com/github/Fatima-TulZahra/AI-Dataset-Advisor/blob/ml%2Fmeta-model/ml/notebooks/01_build_dataset_list.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/ai_dataset_advisor'
for d in ['data/clf', 'data/reg', 'results', 'artifacts', 'openml_cache']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
a = pd.read_csv(f"{BASE}/datasets_list.csv")
b = pd.read_csv(f"{BASE}/datasets_candidates_all.csv")
print("datasets_list.csv:", a.shape)
print("candidates_all   :", b.shape)

datasets_list.csv: (369, 7)
candidates_all   : (1415, 7)


In [4]:
!pip install -q openml

In [5]:
import openml, pandas as pd, re
cc18_ids  = set(openml.study.get_suite(99).data)    # classification collection
ctr23_ids = set(openml.study.get_suite(353).data)   # regression collection

tasks = openml.tasks.list_tasks(output_format="dataframe")   # 1-2 minute lag sakte hain
tasks = tasks[tasks.status == "active"]

bad_names = re.compile(r"BNG|synthetic|random|mnist|cifar|fashion|letter_image", re.I)

def harvest(task_type, is_clf):
    t = tasks[tasks.task_type == task_type]
    t = t[t.NumberOfInstances.between(500, 100000) & t.NumberOfFeatures.between(4, 200)]
    if is_clf:
        t = t[t.NumberOfClasses.between(2, 20) & (t.MinorityClassSize >= 20)]
    t = t[~t.name.str.contains(bad_names)]
    t = t.drop_duplicates("did").drop_duplicates("name")     # ek dataset ek hi baar
    return t

clf = harvest("Supervised Classification", True)
reg = harvest("Supervised Regression", False)
print(len(clf), len(reg))

/usr/local/lib/python3.13/dist-packages/openml/tasks/functions.py:235: RuntimeWarning: Invalid xml for task 362155: 0
From {'oml:task_id': '362155', 'oml:task_type_id': '1', 'oml:task_type': 'Supervised Classification', 'oml:did': '31', 'oml:name': 'credit-g', 'oml:status': 'active', 'oml:format': 'ARFF', 'oml:input': [{'@name': 'estimation_procedure', '#text': '0'}, {'@name': 'source_data', '#text': '31'}, {'@name': 'target_feature', '#text': 'classification problem'}], 'oml:quality': [{'@name': 'MajorityClassSize', '#text': '700.0'}, {'@name': 'MaxNominalAttDistinctValues', '#text': '10.0'}, {'@name': 'MinorityClassSize', '#text': '300.0'}, {'@name': 'NumberOfClasses', '#text': '2.0'}, {'@name': 'NumberOfFeatures', '#text': '21.0'}, {'@name': 'NumberOfInstances', '#text': '1000.0'}, {'@name': 'NumberOfInstancesWithMissingValues', '#text': '0.0'}, {'@name': 'NumberOfMissingValues', '#text': '0.0'}, {'@name': 'NumberOfNumericFeatures', '#text': '7.0'}, {'@name': 'NumberOfSymbolicFeatur

1066 349


In [6]:
def finish(df, task):
    out = pd.DataFrame({
        "did": df.did, "name": df.name, "task": task,
        "n_rows": df.NumberOfInstances, "n_cols": df.NumberOfFeatures,
        "source": df.did.map(lambda d: "CC18" if d in cc18_ids else ("CTR23" if d in ctr23_ids else "OpenML-filter")),
    })
    out["split"] = "train"
    heldout = out.sample(frac=0.15, random_state=42).index
    out.loc[heldout, "split"] = "heldout"
    return out

final = pd.concat([finish(clf, "classification"), finish(reg, "regression")])
final.to_csv(f"{BASE}/datasets_list.csv", index=False)
print(final.groupby(["task", "split"]).size())

task            split  
classification  heldout    160
                train      906
regression      heldout     52
                train      297
dtype: int64


In [8]:
import pandas as pd

full = pd.read_csv(f"{BASE}/datasets_list.csv")
full.to_csv(f"{BASE}/datasets_candidates_all.csv", index=False)   # 1415 ki backup

def pick(df, n_total, seed=42):
    must = df[df.source.isin(["CC18", "CTR23"])]                     # curated wale sab rakho
    rest = df[~df.source.isin(["CC18", "CTR23"])].sample(frac=1, random_state=seed).copy()
    rest["family"] = rest["name"].str.extract(r"^([A-Za-z]+)", expand=False).str.lower()
    rest = rest.groupby("family", group_keys=False).head(4)          # ek series ke max 4
    need = max(n_total - len(must), 0)
    rest["size_bin"] = pd.cut(rest.n_rows, [0, 1500, 5000, 1e12], labels=False)
    parts = [g.sample(min(len(g), need // 3), random_state=seed)
             for _, g in rest.groupby("size_bin")]                   # chhote/darmiyane/bade barabar
    out = pd.concat([must] + parts).drop(columns=["family", "size_bin"], errors="ignore").copy()
    out["split"] = "train"
    out.loc[out.sample(frac=0.15, random_state=seed).index, "split"] = "heldout"
    return out

clf_final = pick(full[full.task == "classification"], 250)
reg_final = pick(full[full.task == "regression"], 120)
final = pd.concat([clf_final, reg_final])
final.to_csv(f"{BASE}/datasets_list.csv", index=False)

print(final.groupby(["task", "split"]).size())
print(final.groupby(["task", "source"]).size())

task            split  
classification  heldout     37
                train      212
regression      heldout     18
                train      102
dtype: int64
task            source       
classification  CC18              51
                OpenML-filter    198
regression      CC18               3
                CTR23             24
                OpenML-filter     93
dtype: int64
